In [ ]:
import json

def count_ids(node):
    if isinstance(node, dict):
        return (1 if "id" in node else 0) + sum(count_ids(v) for v in node.values())
    if isinstance(node, list):
        return sum(count_ids(item) for item in node)
    return 0

with open("Crop_Disease_train_qwenvl.json", "r") as f:
    data = json.load(f)
    print(count_ids(data))


145261


In [ ]:
import os
import json
import time
import random
from google import genai
from google.genai import types
from google.genai.errors import APIError
from tqdm import tqdm
from itertools import islice

INPUT_JSON = "dataset/Crop_Disease_train_qwenvl.json"
OUTPUT_JSON = "cddm_knowledge_graph_triplets.json"

BATCH_SIZE = 20
MODEL_NAME = "gemini-2.0-flash"

client = genai.Client()
print(f"Gemini client initialized with model: {MODEL_NAME}")


def make_batch_prompt(strands):
    strands_json = json.dumps(strands, ensure_ascii=False, indent=2)
    return f"""
    You are an expert knowledge graph architect and data extraction bot specializing in agronomy. 
    Your task is to read a JSON array containing *multiple* conversations about crop diseases and extract all factual information into a *single, flat list* of unique triplets.

    ### Ontology & Schema
    **Node Categories:**
    * `Crop`, `Disease`, `Symptom`, `Pathogen`, `Treatment`

    **Relationship Categories:**
    * `AFFECTS`, `PRESENTS`, `CAUSED_BY`, `TREATED_BY`

    ### Output Format
    Must return a JSON list of triplets: `[subject, relationship, object]`.

    Example:
    [
      ["Powdery Mildew", "AFFECTS", "Cucumber"],
      ["Early Blight", "TREATED_BY", "Copper-based fungicide"]
    ]

    ---
    Here is the batch of conversations to process:
    {strands_json}
    """


def parse_batch_with_gemini(strands):
    if not client:
        print("ERROR: Gemini client not initialized")
        return None

    prompt = make_batch_prompt(strands)

    triplet_list_schema = types.Schema(
        type=types.Type.ARRAY,
        items=types.Schema(
            type=types.Type.ARRAY,
            items=types.Schema(type=types.Type.STRING),
            min_items=3,
            max_items=3
        )
    )

    config = types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
        response_schema=triplet_list_schema,
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=[prompt],
            config=config,
        )
        if response.text:
            return json.loads(response.text)
        else:
            print("⚠️ Empty response from API.")
            return None

    except APIError as e:
        print(f"🚫 API Error: {e}")
        return None
    except json.JSONDecodeError as e:
        raw_response_part = getattr(response, 'text', 'N/A')
        print(f"JSON Decode Error: {e}")
        print(f"Raw response part: {raw_response_part[:200]}...")
        return None
    except Exception as e:
        print(f"💥 Unexpected error: {e}")
        return None


def safe_parse_batch_with_retry(strands, max_retries=5):
    for attempt in range(max_retries):
        results = parse_batch_with_gemini(strands)
        if results is None and attempt < max_retries - 1:
            wait_time = min(60, (2 ** attempt) + random.uniform(0, 2))
            print(f"⏳ Retrying in {wait_time:.1f}s (attempt #{attempt + 1})...")
            time.sleep(wait_time)
            continue
        return results if results is not None else []
    print("❗ Max retries reached for batch.")
    return []


def load_existing_triplets():
    if os.path.exists(OUTPUT_JSON):
        try:
            with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
                triplets = json.load(f)
            print(f"Loaded {len(triplets)} previously saved triplets.")
            return {tuple(t) for t in triplets}
        except Exception as e:
            print(f"Could not load previous file: {e}")
    return set()


def save_triplets_atomic(triplet_set):
    tmp_file = OUTPUT_JSON + ".tmp"
    with open(tmp_file, "w", encoding="utf-8") as f:
        json.dump([list(t) for t in sorted(triplet_set)], f, ensure_ascii=False, indent=2)
    os.replace(tmp_file, OUTPUT_JSON)  
    print(f"💾 Progress saved: {len(triplet_set)} triplets.")




with open(INPUT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)
    print(f"Found {len(data)} total strands in {INPUT_JSON}")

    all_extracted_triplets = load_existing_triplets()

    total_strands = len(data)
    data_iterator = iter(data)

    with tqdm(total=total_strands, desc="Extracting KG Triplets") as pbar:
        processed = 0
        for batch in iter(lambda: list(islice(data_iterator, BATCH_SIZE)), []):
            processed += len(batch)
            triplets = safe_parse_batch_with_retry(batch)

            for triplet in triplets:
                all_extracted_triplets.add(tuple(triplet))

            save_triplets_atomic(all_extracted_triplets)
            time.sleep(1.0)
            pbar.update(len(batch))

    print("\n--- Summary ---")
    print(f"Total strands processed: {total_strands}")
    print(f"Total UNIQUE triplets found: {len(all_extracted_triplets)}")
    print(f"Triplets saved incrementally to {OUTPUT_JSON}")

In [2]:
import json
from tqdm import tqdm
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv
load_dotenv()

JSON_PATH = "../dataset/cddm_knowledge_graph_triplets.json"   

driver = GraphDatabase.driver(os.getenv("NEO4J_URI"), auth=(os.getenv("NEO4J_USER"), os.getenv("NEO4J_PASSWORD")))

with open(JSON_PATH, "r") as f:
    data = json.load(f)

with driver.session() as session:
    for source, relation, target in tqdm(data, desc="Creating graph", unit="rel"):
        relation = relation.upper().replace(" ", "_")
        query = f"""
        MERGE (a:Entity {{name: $source}})
        MERGE (b:Entity {{name: $target}})
        MERGE (a)-[:{relation}]->(b)
        """
        session.run(query, source=source, target=target)

print("Graph built!")
driver.close()

Creating graph: 100%|██████████| 1001/1001 [00:06<00:00, 152.00rel/s]

Graph built!
